In [1]:
import gzip
import pickle
import pandas as pd

In [2]:
from matbench_discovery.data import DataFiles
with gzip.open(DataFiles.mp_patched_phase_diagram.mp_patched_phase_diagram.path, mode="rb") as zip_file:
    ppd_mp = pickle.load(zip_file)

In [3]:
mp_20_csv_data = pd.read_csv("mp_20_test.csv.gz", index_col="material_id")
computed_data = pd.read_csv("mp_20_PBE_54_3.csv.gz", index_col="material_id")
PBE_52_data = pd.read_csv("mp_20_PBE_52.csv.gz", index_col="material_id")
computed_data['mp_20_csv_reference'] = mp_20_csv_data.reindex(computed_data.index).e_above_hull
computed_data['PBE_52'] = PBE_52_data.reindex(computed_data.index).e_above_hull_corrected

In [4]:
from mp_api.client import MPRester
from dotenv import find_dotenv, get_key
MP_API_KEY = get_key(find_dotenv(), "MP_API_KEY")

In [5]:
with MPRester(MP_API_KEY) as mpr:
    for mp_id in computed_data.index:
        reference_records = mpr.get_entry_by_material_id(mp_id)
        for record in reference_records:
            found = False
            if record.data['run_type'] in ("GGA", "GGA_U"):
                computed_data.loc[mp_id, "mp_e_above_hull"] = ppd_mp.get_e_above_hull(record, allow_negative=True)
                computed_data.loc[mp_id, "mp_e_uncorrected"] = record.uncorrected_energy
                if found:
                    raise ValueError(f"Multiple records found for {mp_id}")
                found = True

Retrieving ThermoDoc documents:   0%|          | 0/2 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/3 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/3 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/2 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/2 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/3 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/2 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/2 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/2 [00:00<?, ?it/s]

Retrieving ThermoDoc documents:   0%|          | 0/3 [00:00<?, ?it/s]

In [6]:
from pymatgen.entries.computed_entries import ComputedEntry
from ast import literal_eval
computed_data["PBE_54_energy_uncorrected"] = computed_data.entries.map(
    lambda x: ComputedEntry.from_dict(literal_eval(x)).uncorrected_energy)
computed_data.loc["mp-19449", "PBE_54_energy_uncorrected_script"] = -82.51573047
computed_data.loc["mp-752737", "PBE_54_energy_uncorrected_script"] = -77.40705622
computed_data.drop(columns=["entries", "structure"])

,e_above_hull_corrected,mp_20_csv_reference,PBE_52,mp_e_above_hull,mp_e_uncorrected,PBE_54_energy_uncorrected,PBE_54_energy_uncorrected_script
material_id,,,,,,,
mp-19449,0.045934,0.000000,0.045934,-6.217249e-15,-83.158863,-82.515789,-82.515730
mp-28599,-0.001522,0.000316,-0.001522,2.612457e-04,-23.581418,-23.593901,NaN
mp-1030119,-0.018732,0.002073,-0.019517,2.072728e-03,-90.783443,-91.033103,NaN
mp-1221948,0.131872,0.044558,0.131872,4.455771e-02,-31.109103,-30.759844,NaN
mp-1103947,0.011350,0.000261,0.011350,-7.124310e+00,-62.119994,-61.964743,NaN
mp-864930,0.022913,0.000000,0.022913,0.000000e+00,-30.122192,-30.030538,NaN
mp-752737,0.159963,0.067626,NaN,6.763063e-02,-78.515757,-77.407773,-77.407056
mp-1516349,0.031824,0.028802,NaN,4.036693e-02,-74.691233,-74.776666,NaN
mp-1220086,0.135953,0.019049,0.135953,-8.739883e+00,-92.506348,-91.103512,NaN
